In [ ]:
import argparse
import csv
import json
import sys
from pathlib import Path
from urllib.parse import quote
from typing import Optional


import re
import time
import requests

ENSEMBL_REST_SERVER = "https://rest.ensembl.org"

_VEP_TRANSIENT_STATUS = {429, 500, 502, 503, 504}


def _vep_call(variant_str, species, max_retries=4, backoff_base=2.0):
    """
    Retries transient errors (429/500/502/503/504, timeouts, connection
    resets) with exponential backoff (2s, 4s, 8s...) before giving up.
    Ensembl's public REST server intermittently 500s/times-out under
    sustained sequential load -- this is a transient server
    condition, not a problem with the input, since retrying the same HGVS
    later typically succeeds. This is distinct from a 400
    (a real HGVS-format rejection), which is NOT retried here -- that's
    handled by _build_hgvs_candidates trying a different INPUT FORM instead,
    in fetch_vep_annotation below.
    """
    encoded = quote(variant_str, safe="")
    endpoint = f"/vep/{species}/hgvs/{encoded}?canonical=1&hgvs=1&af=1&REVEL=1&SpliceAI=1"
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    url = ENSEMBL_REST_SERVER + endpoint

    resp = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, headers=headers, timeout=30)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as exc:
            if attempt == max_retries:
                raise
            wait = backoff_base ** attempt
            print(f"  VEP network error ({exc}); retrying in {wait:.0f}s ({attempt}/{max_retries})")
            time.sleep(wait)
            continue

        if resp.status_code in _VEP_TRANSIENT_STATUS and attempt < max_retries:
            wait = backoff_base ** attempt
            print(f"  VEP HTTP {resp.status_code}; retrying in {wait:.0f}s ({attempt}/{max_retries})")
            time.sleep(wait)
            continue

        return resp

    return resp


def _vep_error_text(resp):
    try:
        return resp.json().get("error", "")
    except Exception:
        return resp.text[:300] if resp.text else ""


def _build_hgvs_candidates(hgvs_variant):
    """
    VEP rejects some valid HGVS forms. Build a fallback list:
      1. Original input
      2. Without gene name in parentheses  (NM_X.Y(GENE):c.Z  -> NM_X.Y:c.Z)
      3. Without NM version number          (NM_X.Y:c.Z        -> NM_X:c.Z)
      4. Both stripped
    """
    candidates = [hgvs_variant]

    no_gene = re.sub(r"\([^)]+\)", "", hgvs_variant)
    if no_gene != hgvs_variant:
        candidates.append(no_gene)

    no_version = re.sub(r"(NM_\d+)\.\d+", r"\1", hgvs_variant)
    if no_version not in candidates:
        candidates.append(no_version)

    no_gene_no_ver = re.sub(r"(NM_\d+)\.\d+", r"\1", no_gene)
    if no_gene_no_ver not in candidates:
        candidates.append(no_gene_no_ver)

    return candidates


def fetch_vep_annotation(hgvs_variant: str, species: str = "human"):
    """
    Query Ensembl VEP REST API for one HGVS variant.

    Automatically retries with progressively simplified HGVS forms if VEP
    returns 400 (the gene name in parentheses is a common cause of 400 errors
    even when the HGVS itself is correct).
    """
    candidates = _build_hgvs_candidates(hgvs_variant)
    last_error = ""

    for attempt, variant_str in enumerate(candidates, 1):
        if attempt > 1:
            print(f"  VEP retry {attempt}: trying stripped form '{variant_str}'")

        resp = _vep_call(variant_str, species)

        if resp.status_code == 200:
            data = resp.json()
            return data[0] if data else None

        last_error = _vep_error_text(resp)

        if resp.status_code == 400:
            print(f"  VEP 400 for '{variant_str}': {last_error or 'Bad Request'}")
            continue   # try next candidate

        # Non-400 HTTP error — surface it immediately
        resp.raise_for_status()

    # All candidates exhausted
    print(f"\n[VEP] All input forms rejected for: {hgvs_variant}")
    print(f"[VEP] Last error: {last_error}")
    print("Hints:")
    print("  • Substitutions use  >  not underscore:  c.100A>T")
    print("  • Intronic positions use minus sign:  c.100-1G>A")
    print("  • Verify NM accession at https://www.ncbi.nlm.nih.gov/nuccore/")
    print("  • Try without version: NM_000256(MYBPC3):c.1208-1G>A")
    return None


In [ ]:
GNOMAD_API = "https://gnomad.broadinstitute.org/api"
from functools import lru_cache
ENSEMBL_SEQUENCE_API = "https://rest.ensembl.org/sequence/region/human"

def _revcomp(seq):
    return seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]


def _fetch_reference_sequence(chrom, start, end):
    """
    GRCh38 reference sequence for chrom:start-end (1-based inclusive),
    always plus-strand -- used to build a correctly-anchored gnomAD/VCF id
    for a pure indel without relying on VEP's allele_string letters, which
    are ambiguous for indels on a minus-strand gene (see
    build_gnomad_variant_id below).
    """
    try:
        resp = requests.get(
            f"{ENSEMBL_SEQUENCE_API}/{chrom}:{start}-{end}",
            headers={"Content-Type": "application/json"},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json().get("seq")
    except Exception as exc:
        print(f"Ensembl sequence lookup warning ({chrom}:{start}-{end}): {exc}")
        return None


def build_gnomad_variant_id(vep_result):
    """
    Build a gnomAD variant ID (chrom-pos-ref-alt, GRCh38, always reference/
    plus-strand orientation -- the convention gnomAD and VCF-style AC/AN
    dumps use) from a VEP result. VEP encodes allele_string as "ref/alt".

    IMPORTANT: when VEP is queried with a transcript-based HGVS input (e.g.
    "NM_000256.3:c.3811C>T") on a minus-strand gene, its top-level
    allele_string is reported in the TRANSCRIPT's orientation, not the
    genomic plus strand - confirmed by VEP itself rejecting the naively
    "matching" genomic HGVS ("14:g.23432713G>A" errors with "Reference
    allele extracted... (C) does not match... (G)"; the true plus-strand
    base is C). Of the 8 ClinGen Cardiomyopathy VCEP genes, 7 (all except
    TPM1) are minus-strand, so this silently produced a wrong id for nearly
    every gene in the panel whenever a c. HGVS variant was entered - which
    is the pipeline's own documented input format. Any code keying a lookup
    (gnomAD FAF95/AC-AN, SpliceAI, case-cohort TSVs) off this id was affected.

    Fix (SNVs): reverse-complement ref/alt when the top-level "strand" is
    -1, so the returned id is always genomic plus-strand regardless of
    which strand the query transcript happened to be on.

    Pure insertions/deletions ("ref/-" or "-/alt" in allele_string) need a
    SEPARATE fix, confirmed as a real bug (found via the 57-variant MYH7
    validation set): naively formatting "-" as a literal empty allele
    produces a malformed, unparseable id (e.g. "14-23424907-AAG--" -- 5
    dash-separated parts, not 4), silently breaking every downstream
    chrom/pos-keyed lookup for that variant (PM4's repeat-region check,
    PS4's cohort-file lookup, gnomAD control AC/AN) even though VEP's own
    `start`/`end` were correct all along. VCF/gnomAD has no representation
    for an empty allele -- both ref and alt need one shared flanking
    reference base prepended (position shifts one base left).

    For a DELETION, the deleted bases in allele_string are ALSO in
    transcript orientation on a minus-strand gene (e.g. c.2539_2541delAAG on MYH7: allele_string reports "AAG", but the real
    plus-strand reference at that exact genomic range is "CTT", its
    reverse complement). Revcomp-ing the letters would work, but directly
    fetching the true reference sequence for the deleted range sidesteps
    the ambiguity entirely and is what's done here.

    For an INSERTION, the inserted bases are novel sequence absent from the
    reference genome, so they can't be looked up this way - those genuinely
    do need reverse-complementing when the query transcript is minus-strand,
    same logic as the existing SNV fix above.

    This produces the minimal, standard single-flanking-base VCF
    representation. It is NOT guaranteed to byte-match gnomAD's own stored
    representation for an indel sitting in a repeat/microsatellite region,
    where multiple equally-valid normalizations exist (confirmed: gnomAD's
    API requires its own exact stored allele string and does not
    fuzzy-match/re-normalize on lookup) - in that case gnomAD-specific
    lookups will report "not found" rather than crash, which is an honest,
    documented limitation rather than a silent wrong answer.
    """
    chrom = str(vep_result.get("seq_region_name", "")).strip()
    pos   = vep_result.get("start")
    end   = vep_result.get("end")
    allele_str = str(vep_result.get("allele_string", ""))
    strand = vep_result.get("strand")

    if not (chrom and pos and "/" in allele_str):
        return None

    parts = allele_str.split("/")
    if len(parts) != 2:
        return None

    ref, alt = parts[0], parts[1]

    if ref == "-" or alt == "-":
        anchor_pos = int(pos) - 1
        anchor_base = _fetch_reference_sequence(chrom, anchor_pos, anchor_pos)
        if anchor_base is None:
            return None

        if alt == "-":
            if end is None:
                return None
            deleted_seq = _fetch_reference_sequence(chrom, int(pos), int(end))
            if deleted_seq is None:
                return None
            return f"{chrom}-{anchor_pos}-{anchor_base + deleted_seq}-{anchor_base}"

        inserted_seq = _revcomp(alt) if (strand == -1 and set(alt) <= set("ACGT")) else alt
        return f"{chrom}-{anchor_pos}-{anchor_base}-{anchor_base + inserted_seq}"

    if strand == -1 and set(ref) <= set("ACGT") and set(alt) <= set("ACGT"):
        ref, alt = _revcomp(ref), _revcomp(alt)

    return f"{chrom}-{pos}-{ref}-{alt}"


_GNOMAD_TRANSIENT_STATUS = {429, 500, 502, 503, 504}
_gnomad_variant_data_cache = {}


def fetch_gnomad_variant_data(gnomad_variant_id, dataset="gnomad_r4", max_retries=6, backoff_base=2.5):
    """
   ONE combined GraphQL query for every gnomAD-derived field this pipeline
    needs per variant: joint AC/AN (PS4 control arm, via fetch_gnomad_ac_an
    in classifierr.ipynb) and genome/exome FAF95 (BA1/BS1/PM2, via
    fetch_gnomad_faf below).

    Two deliberate choices, both from reproduced bugs on the 57-variant MYH7
    validation batch:

    - One request per variant, not the original two: two calls per variant
    sustained enough volume to trip gnomAD rate limiting mid-batch (FAF95
    came back None for ~45% of variants) while isolated requests still
    succeeded in <1s. The module-level cache lets the two caller functions,
    invoked in either order by classify_variant, share that one call.

    - Manual cache, not @lru_cache: only a genuine HTTP response is cached
    (a GraphQL "Variant not found" counts -- it's a stable answer). A
    network failure after all retries returns `empty` UNcached, so the
    next caller gets a fresh attempt instead of inheriting a dead result.

    Retries transient errors (429/500/502/503/504, timeouts, connection
    resets) with exponential backoff, same as _vep_call.
    Returns raw parsed JSON ({"data", "errors"}) for callers to extract.
    """
    empty = {"data": None, "errors": None}
    if not gnomad_variant_id:
        return empty

    cache_key = (gnomad_variant_id, dataset)
    if cache_key in _gnomad_variant_data_cache:
        return _gnomad_variant_data_cache[cache_key]

    query = """
    query CombinedVariantData($variantId: String!, $dataset: DatasetId!) {
      variant(variantId: $variantId, dataset: $dataset) {
        joint { ac an }
        genome { faf95 { popmax popmax_population } }
        exome { faf95 { popmax popmax_population } }
      }
    }
    """

    resp = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(
                GNOMAD_API,
                json={"query": query, "variables": {"variantId": gnomad_variant_id, "dataset": dataset}},
                headers={"Content-Type": "application/json"},
                timeout=90,  # some variants' 'joint' aggregation is slow server-side (observed ~50s)
            )
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as exc:
            if attempt == max_retries:
                print(f"  gnomAD lookup failed for {gnomad_variant_id}: {type(exc).__name__}: {exc}")
                return empty  # NOT cached -- next call gets a fresh retry
            wait = backoff_base ** attempt
            print(f"  gnomAD network error ({exc}); retrying in {wait:.0f}s ({attempt}/{max_retries})")
            time.sleep(wait)
            continue

        if resp.status_code in _GNOMAD_TRANSIENT_STATUS and attempt < max_retries:
            wait = backoff_base ** attempt
            print(f"  gnomAD HTTP {resp.status_code}; retrying in {wait:.0f}s ({attempt}/{max_retries})")
            time.sleep(wait)
            continue

        break

    if resp is None:
        return empty  # NOT cached

    try:
        resp.raise_for_status()
        data = resp.json()
    except Exception as exc:
        body = resp.text[:300] if resp is not None and resp.text else ""
        print(f"  gnomAD lookup failed for {gnomad_variant_id}: "
              f"{type(exc).__name__}: {exc}" + (f" -- response body: {body}" if body else ""))
        return empty  # NOT cached

    _gnomad_variant_data_cache[cache_key] = data  # a real response -- cache it
    return data


def fetch_gnomad_faf(gnomad_variant_id, dataset="gnomad_r4"):
    """
    FAF95 (filter allele frequency, 95% CI lower bound) -- the highest
    popmax across genome/exome sources. Sourced from the shared, cached
    fetch_gnomad_variant_data (see its docstring for why FAF95 and PS4's
    AC/AN were merged into one request).
    """
    empty = {"faf95_popmax": None, "faf95_population": None, "faf_source": None}

    data = fetch_gnomad_variant_data(gnomad_variant_id, dataset)
    if data.get("errors"):
        return empty

    variant_data = (data.get("data") or {}).get("variant")
    if not variant_data:
        return empty

    best_faf, best_pop, best_source = None, None, None

    for source in ("genome", "exome"):
        faf95 = ((variant_data.get(source) or {}).get("faf95") or {})
        popmax = faf95.get("popmax")
        if popmax is None:
            continue
        try:
            popmax = float(popmax)
        except (TypeError, ValueError):
            continue
        if best_faf is None or popmax > best_faf:
            best_faf  = popmax
            best_pop  = faf95.get("popmax_population")
            best_source = source

    return {"faf95_popmax": best_faf, "faf95_population": best_pop, "faf_source": best_source}

In [6]:

def extract_frequency_fields(obj):
    """
    Recursively pull out fields that look like population frequency evidence.
    """
    if isinstance(obj, dict):
        out = {}
        for key, value in obj.items():
            key_l = key.lower()
            if any(token in key_l for token in ("af", "freq", "gnomad")):
                out[key] = value
            elif isinstance(value, (dict, list)):
                nested = extract_frequency_fields(value)
                if nested not in (None, {}, []):
                    out[key] = nested
        return out

    if isinstance(obj, list):
        items = []
        for item in obj:
            nested = extract_frequency_fields(item)
            if nested not in (None, {}, []):
                items.append(nested)
        return items

    return None


def build_json_output(result: dict) -> dict:
    """
    Build the structured JSON output that the classifier can consume.
    """
    transcripts = result.get("transcript_consequences", [])

    transcript_entries = []
    for tx in transcripts:
        transcript_entries.append(
            {
                "transcript_id": tx.get("transcript_id"),
                "gene_id": tx.get("gene_id"),
                "gene_symbol": tx.get("gene_symbol"),
                "biotype": tx.get("biotype"),
                "canonical": tx.get("canonical"),
                "impact": tx.get("impact"),
                "consequence_terms": tx.get("consequence_terms", []),
                "hgvsc": tx.get("hgvsc"),
                "hgvsp": tx.get("hgvsp"),
                "protein_id": tx.get("protein_id"),
                "codons": tx.get("codons"),
                "amino_acids": tx.get("amino_acids"),
                "strand": tx.get("strand"),
                "cdna_position": tx.get("cdna_position"),
                "cds_position": tx.get("cds_position"),
                "protein_position": tx.get("protein_position"),
            }
        )

    return {
        "input": result.get("input"),
        "most_severe_consequence": result.get("most_severe_consequence"),
        "population_frequencies": extract_frequency_fields(result),
        "transcripts": transcript_entries,
    }


def build_table_rows(output_json: dict) -> list[dict]:
    """
    Flatten the JSON output into one row per transcript for TSV output.
    """
    rows = []
    for tx in output_json.get("transcripts", []):
        rows.append(
            {
                "input": output_json.get("input"),
                "most_severe_consequence": output_json.get("most_severe_consequence"),
                "transcript_id": tx.get("transcript_id"),
                "gene_id": tx.get("gene_id"),
                "gene_symbol": tx.get("gene_symbol"),
                "biotype": tx.get("biotype"),
                "canonical": tx.get("canonical"),
                "impact": tx.get("impact"),
                "consequence_terms": ",".join(tx.get("consequence_terms", [])),
                "hgvsc": tx.get("hgvsc"),
                "hgvsp": tx.get("hgvsp"),
                "protein_id": tx.get("protein_id"),
                "codons": tx.get("codons"),
                "amino_acids": tx.get("amino_acids"),
                "strand": tx.get("strand"),
                "cdna_position": tx.get("cdna_position"),
                "cds_position": tx.get("cds_position"),
                "protein_position": tx.get("protein_position"),
            }
        )
    return rows




In [7]:
def write_json_file(output_json: dict, output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(output_json, f, indent=2)


def write_tsv_file(rows: list[dict], output_path: Path) -> None:
    if not rows:
        output_path.write_text("", encoding="utf-8")
        return

    fieldnames = list(rows[0].keys())
    with output_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)


def main() -> int:
    parser = argparse.ArgumentParser(
        description="Annotate a single HGVS variant using Ensembl VEP REST API."
    )
    parser.add_argument(
        "variant",
        help="HGVS variant string, for example ENST00000003084:c.1431_1433delTTC",
    )
    parser.add_argument(
        "--output-dir",
        default="outputs",
        help="Directory for generated files (default: outputs)",
    )
    parser.add_argument(
        "--basename",
        default="vep_annotation",
        help="Base name for output files (default: vep_annotation)",
    )
    args = parser.parse_args()

    try:
        result = fetch_vep_annotation(args.variant)
        if result is None:
            print("Could not find variant info.")
            return 1

        output_json = build_json_output(result)
        table_rows = build_table_rows(output_json)

        output_dir = Path(args.output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        json_path = output_dir / f"{args.basename}.json"
        tsv_path = output_dir / f"{args.basename}.tsv"

        write_json_file(output_json, json_path)
        write_tsv_file(table_rows, tsv_path)

        print(f"Saved JSON to: {json_path}")
        print(f"Saved table to: {tsv_path}")
        print(f"Transcripts found: {len(table_rows)}")

        # Helpful debug output
        print("Most severe consequence:", output_json.get("most_severe_consequence"))
        if table_rows:
            print("First transcript row:")
            print(table_rows[0])

        return 0

    except requests.HTTPError as e:
        print(f"HTTP error from VEP API: {e}", file=sys.stderr)
        return 1
    except requests.RequestException as e:
        print(f"Request error: {e}", file=sys.stderr)
        return 1
    except Exception as e:
        print(f"Unexpected error: {e}", file=sys.stderr)
        return 1

'''
if __name__ == "__main__":
    raise SystemExit(main())

'''

'\nif __name__ == "__main__":\n    raise SystemExit(main())\n\n'

In [ ]:
import json

def _extract_revel_from_vep(result):
    """REVEL score from VEP's own REVEL plugin (?REVEL=1), off the chosen transcript."""
    tx = choose_transcript(result) or {}
    score = tx.get("revel")
    return float(score) if score is not None else None


def _extract_spliceai_from_vep(result):
    """Max SpliceAI delta score (acceptor/donor gain/loss) from VEP's SpliceAI
    plugin (?SpliceAI=1); scores are variant-level so any transcript's copy works."""
    tx = choose_transcript(result) or {}
    deltas = tx.get("spliceai") or {}
    scores = [deltas.get(k) for k in ("DS_AG", "DS_AL", "DS_DG", "DS_DL")]
    scores = [float(s) for s in scores if s is not None]
    return max(scores) if scores else None


def choose_transcript(result):
    """
    Prefer the canonical transcript if one is present.
    Otherwise return the first transcript consequence.
    """
    transcripts = result.get("transcript_consequences", []) or []
    if not transcripts:
        return None

    canonical = [tx for tx in transcripts if tx.get("canonical") == 1]
    return canonical[0] if canonical else transcripts[0]


def extract_matching_fields(obj, keywords):
    """
    Recursively extract fields whose key names contain any of the keywords.
    Returns a flat dict of path -> value.
    """
    matches = {}

    def walk(x, path=""):
        if isinstance(x, dict):
            for k, v in x.items():
                key_l = str(k).lower()
                new_path = f"{path}.{k}" if path else str(k)
                if any(word in key_l for word in keywords):
                    if not isinstance(v, (dict, list)):
                        matches[new_path] = v
                if isinstance(v, (dict, list)):
                    walk(v, new_path)
        elif isinstance(x, list):
            for i, item in enumerate(x):
                walk(item, f"{path}[{i}]")

    walk(obj)
    return matches


def extract_gnomad_frequencies(result):
    colocated = result.get("colocated_variants", []) or []
    extracted = []

    for cv in colocated:
        if not isinstance(cv, dict):
            continue

        freqs = cv.get("frequencies")
        if freqs:
            extracted.append(
                {
                    "id": cv.get("id"),
                    "allele_string": cv.get("allele_string"),
                    "frequencies": freqs,
                }
            )

    return extracted


def get_max_population_af(result):
    """
    Extract the global (overall) gnomAD allele frequency from VEP colocated_variants.

    ClinGen ACMG rules require the GLOBAL (all-population) AF, not the max across
    individual subpopulations. Using a population-specific maximum (e.g. SAS) inflates
    the apparent frequency and can incorrectly trigger BS1 for ultra-rare variants.

    Priority:
      1. gnomadg / gnomade   — global gnomAD genome / exome (preferred)
      2. gnomad               — generic gnomAD key
      3. Max across all keys  — fallback if no global key present
    """
    GLOBAL_KEYS = {"gnomadg", "gnomade", "gnomad", "af"}

    best_global_af  = None
    best_global_src = None
    best_any_af     = None
    best_any_src    = None

    for cv in (result.get("colocated_variants") or []):
        freqs = cv.get("frequencies", {})
        for allele, allele_freqs in freqs.items():
            if not isinstance(allele_freqs, dict):
                continue
            for source, value in allele_freqs.items():
                try:
                    value = float(value)
                except (TypeError, ValueError):
                    continue
                if value == 0:
                    continue
                # Track global keys separately
                if source.lower() in GLOBAL_KEYS:
                    if best_global_af is None or value > best_global_af:
                        best_global_af  = value
                        best_global_src = source
                # Track overall maximum as fallback
                if best_any_af is None or value > best_any_af:
                    best_any_af  = value
                    best_any_src = source

    if best_global_af is not None:
        return {"max_population_af": best_global_af, "max_population_source": best_global_src}

    return {"max_population_af": best_any_af, "max_population_source": best_any_src}

def find_keys_containing(obj, keywords):
    found = {}

    def walk(x, path=""):
        if isinstance(x, dict):
            for k, v in x.items():
                new_path = f"{path}.{k}" if path else k
                if any(word in k.lower() for word in keywords):
                    found[new_path] = v
                walk(v, new_path)
        elif isinstance(x, list):
            for i, item in enumerate(x):
                walk(item, f"{path}[{i}]")

    walk(obj)
    return found



def extract_clinvar_significance(result):
    """
    Extract ClinVar clinical significance from VEP colocated_variants.
    Returns the most severe classification found, or None.
    Priority: pathogenic > likely_pathogenic > uncertain_significance > likely_benign > benign
    """
    priority = {
        "pathogenic": 0, "likely_pathogenic": 1,
        "uncertain_significance": 2, "likely_benign": 3, "benign": 4,
    }
    best = None
    for cv in (result.get("colocated_variants") or []):
        for sig in (cv.get("clin_sig") or []):
            sig_norm = sig.lower().replace(" ", "_")
            if best is None or priority.get(sig_norm, 99) < priority.get(best, 99):
                best = sig_norm
    return best

def build_json_output(result):
    transcripts = result.get("transcript_consequences", []) or []

    transcript_entries = []
    for tx in transcripts:
        transcript_entries.append({
            "gene_symbol":       tx.get("gene_symbol"),
            "gene_id":           tx.get("gene_id"),
            "transcript_id":     tx.get("transcript_id"),
            "canonical":         tx.get("canonical"),
            "biotype":           tx.get("biotype"),
            "impact":            tx.get("impact"),
            "consequence_terms": tx.get("consequence_terms", []),
            "hgvsc":             tx.get("hgvsc"),
            "hgvsp":             tx.get("hgvsp"),
            "protein_position":  tx.get("protein_position"),
            "amino_acids":       tx.get("amino_acids"),
            "codons":            tx.get("codons"),
            "strand":            tx.get("strand"),
        })

    # VEP-derived max AF
    pop_summary = get_max_population_af(result)

    # gnomAD FAF95
    gnomad_id = build_gnomad_variant_id(result)
    faf       = fetch_gnomad_faf(gnomad_id)
    pop_summary.update(faf)
    pop_summary["gnomad_variant_id"] = gnomad_id

    # REVEL + SpliceAI: requested directly from VEP's own plugins
    # (?REVEL=1&SpliceAI=1 in _vep_call) instead of separate live calls to
    # MyVariant.info / the Broad SpliceAI Lookup API.
    revel_score    = _extract_revel_from_vep(result)
    spliceai_score = _extract_spliceai_from_vep(result)

    return {
        "input":                     result.get("input"),
        "most_severe_consequence":   result.get("most_severe_consequence"),
        "allele_string":             result.get("allele_string"),
        "population_summary":        pop_summary,
        "transcripts":               transcript_entries,
        "gnomad_frequencies":        extract_gnomad_frequencies(result),
        "revel_score":               revel_score,
        "revel":                     find_keys_containing(result, ["revel"]),
        "spliceai_score":            spliceai_score,
        "spliceai":                  find_keys_containing(result, ["spliceai"]),
        "clinvar_significance":      extract_clinvar_significance(result),
    }


def build_table_rows(output_json):
    """
    One row per transcript.
    """
    rows = []
    pop = output_json.get("population_summary", {})

    for tx in output_json.get("transcripts", []):
        rows.append({
            "input":                    output_json.get("input"),
            "most_severe_consequence":  output_json.get("most_severe_consequence"),
            "gene_symbol":              tx.get("gene_symbol"),
            "gene_id":                  tx.get("gene_id"),
            "transcript_id":            tx.get("transcript_id"),
            "canonical":                tx.get("canonical"),
            "biotype":                  tx.get("biotype"),
            "impact":                   tx.get("impact"),
            "consequence_terms":        ",".join(tx.get("consequence_terms", [])),
            "hgvsc":                    tx.get("hgvsc"),
            "hgvsp":                    tx.get("hgvsp"),
            "protein_position":         tx.get("protein_position"),
            "amino_acids":              tx.get("amino_acids"),
            "codons":                   tx.get("codons"),
            # population
            "max_population_af":        pop.get("max_population_af"),
            "max_population_source":    pop.get("max_population_source"),
            "gnomad_variant_id":        pop.get("gnomad_variant_id"),
            "faf95_popmax":             pop.get("faf95_popmax"),
            "faf95_population":         pop.get("faf95_population"),
            "faf_source":               pop.get("faf_source"),
            # in silico
            "revel_score":              output_json.get("revel_score"),
            "spliceai_score":           output_json.get("spliceai_score"),
            "gnomad_frequencies":       json.dumps(output_json.get("gnomad_frequencies", [])),
            "spliceai":                 json.dumps(output_json.get("spliceai", {})),
        })

    return rows


In [11]:
from pathlib import Path

def write_json_file(output_json: dict, output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(output_json, f, indent=2)


def write_tsv_file(rows: list[dict], output_path: Path) -> None:
    if not rows:
        output_path.write_text("", encoding="utf-8")
        return

    fieldnames = list(rows[0].keys())
    with output_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)

In [12]:
def annotate_variant(variant):
    result = fetch_vep_annotation(variant)
    if result is None:
        return None, None

    output_json = build_json_output(result)
    table_rows = build_table_rows(output_json)
    return output_json, table_rows

In [13]:
def save_annotation_outputs(output_json, table_rows, outdir="outputs", basename="variant"):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    json_path = outdir / f"{basename}.json"
    tsv_path = outdir / f"{basename}.tsv"

    write_json_file(output_json, json_path)
    write_tsv_file(table_rows, tsv_path)

    return json_path, tsv_path